In [1]:
import dask.dataframe as dd

# Load the CSV
spot_df = dd.read_csv("_NIFTY_IDX__202507041318.csv", usecols=["Date", "Time", "Open", "High", "Low", "Close"])

# Convert Date and Time to a single DateTime column
spot_df["Datetime"] = dd.to_datetime(spot_df["Date"] + " " + spot_df["Time"])

# Drop the original Date and Time columns
spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

# Compute to check the first few rows
print(spot_df.head())


/home/newberry3/venv/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20


In [2]:
import dask
import dask.dataframe as dd
import pickle
import glob

# Path to your options pickle files
options_files = glob.glob("/home/newberry3/Data/NIFTY/*.pkl")  # Assuming all pickles have .pkl extension # Adjust path as needed to match all files

# Function to load a single pickle file
def load_pickle(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)  # Assuming each pickle file contains a DataFrame

# Load all pickles using dask.delayed
delayed_dfs = [dask.delayed(load_pickle)(file) for file in options_files]

# Convert delayed objects to Dask DataFrame
options_df = dd.from_delayed(delayed_dfs)

# Compute to check structure
print(options_df.head())


         Date   Time  ExpiryDate StrikePrice Type     Open     High      Low  \
0  2024-01-01  09:16  2024-01-04       20000   CE  1770.45  1770.45  1770.45   
1  2024-01-01  09:19  2024-01-04       20000   CE  1718.00  1718.00  1718.00   
2  2024-01-01  09:20  2024-01-04       20000   CE  1718.00  1718.00  1718.00   
3  2024-01-01  09:21  2024-01-04       20000   CE  1720.00  1720.00  1720.00   
4  2024-01-01  09:29  2024-01-04       20000   CE  1720.00  1720.00  1720.00   

     Close                           Ticker  
0  1770.45  2024010109:16NIFTY24010420000CE  
1  1718.00  2024010109:19NIFTY24010420000CE  
2  1718.00  2024010109:20NIFTY24010420000CE  
3  1720.00  2024010109:21NIFTY24010420000CE  
4  1720.00  2024010109:29NIFTY24010420000CE  


In [3]:
# Save spot data to Parquet
spot_df.to_parquet("nifty_spot_data.parquet", write_index=False)

# Save options data to Parquet
#options_df.to_parquet("nifty_options_data.parquet", write_index=False)

# Reload data from Parquet to verify
#spot_df = dd.read_parquet("nifty_spot_data.parquet")
#options_df = dd.read_parquet("nifty_options_data.parquet")

# Check structure
print(spot_df.head())
#print(options_df.head())


             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20


In [4]:
import pandas as pd

# Load spot data from Parquet
spot_df = pd.read_parquet("nifty_spot_data.parquet")

# Ensure Datetime is in datetime format and sorted
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')

# Set datetime as index
spot_df.set_index('Datetime', inplace=True)

# Filter data to start from 9:00 AM and resample to 15-minute OHLC
spot_df = spot_df.between_time('09:15:00', '15:29:00')

# Resample to 15-minute OHLC
spot_5min = spot_df.resample('5T').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# Reset index for usability
spot_5min = spot_5min.reset_index()

# Optional: Save for later use
spot_5min.to_parquet("nifty_spot_data_5min.parquet", index=False)

# Preview
print(spot_5min.head())  # Shows the first few rows including the 9:00 AM candle


             Datetime      Open      High       Low     Close
0 2021-06-01 09:15:00  15630.30  15630.30  15588.65  15602.95
1 2021-06-01 09:20:00  15601.55  15640.25  15600.85  15639.50
2 2021-06-01 09:25:00  15637.55  15650.15  15636.05  15640.00
3 2021-06-01 09:30:00  15640.60  15655.95  15637.90  15648.85
4 2021-06-01 09:35:00  15650.15  15652.60  15644.55  15647.80


/tmp/ipykernel_38180/1748993910.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  spot_5min = spot_df.resample('5T').agg({


daily

In [5]:
import pandas as pd

# Load intraday spot data
spot_df = pd.read_parquet("nifty_spot_data_5min.parquet")

# Ensure datetime format and sorting
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')
spot_df.set_index('Datetime', inplace=True)

# Filter market hours (optional, if needed)
spot_df = spot_df.between_time('09:15:00', '15:30:00')

# Resample to daily OHLC
spot_daily = spot_df.resample('1D').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna().reset_index()

# === Calculate 20-day EMA on daily close ===
spot_daily['EMA_20'] = spot_daily['Close'].ewm(span=20, adjust=False).mean()

# === Optional: Save daily data with EMA ===
spot_daily.to_csv("spot_daily_with_ema.csv", index=False)

# === Preview ===
print(spot_daily.tail())


      Datetime      Open      High       Low     Close        EMA_20
993 2025-06-09  25127.20  25143.45  25077.15  25102.15  24713.692837
994 2025-06-10  25199.30  25199.30  25055.45  25091.50  24749.674471
995 2025-06-11  25136.20  25222.40  25081.30  25125.75  24785.491188
996 2025-06-12  25164.50  25196.20  24825.90  24856.00  24792.206313
997 2025-06-13  24509.05  24754.35  24508.10  24735.55  24786.810474


In [6]:
import pandas as pd

# Load your 5-min data and daily EMA data
spot_5min = pd.read_parquet("nifty_spot_data_5min.parquet")
daily_ema = pd.read_csv("spot_daily_with_ema.csv", parse_dates=['Datetime'])  # Replace with your actual file

# Add a date column to both for merging
spot_5min['Date'] = pd.to_datetime(spot_5min['Datetime']).dt.date
daily_ema['Date'] = pd.to_datetime(daily_ema['Datetime']).dt.date

# Merge the daily EMA into the 5-minute data
spot_5min_merged = pd.merge(spot_5min, daily_ema[['Date', 'EMA_20']], on='Date', how='left')

# Optional: drop the temporary 'Date' column if not needed
# spot_5min_merged.drop(columns=['Date'], inplace=True)

# Save or preview
spot_5min_merged.to_parquet("spot_5min_with_daily_ema.parquet", index=False)
print(spot_5min_merged.head(100))



              Datetime      Open      High       Low     Close        Date  \
0  2021-06-01 09:15:00  15630.30  15630.30  15588.65  15602.95  2021-06-01   
1  2021-06-01 09:20:00  15601.55  15640.25  15600.85  15639.50  2021-06-01   
2  2021-06-01 09:25:00  15637.55  15650.15  15636.05  15640.00  2021-06-01   
3  2021-06-01 09:30:00  15640.60  15655.95  15637.90  15648.85  2021-06-01   
4  2021-06-01 09:35:00  15650.15  15652.60  15644.55  15647.80  2021-06-01   
..                 ...       ...       ...       ...       ...         ...   
95 2021-06-02 10:55:00  15493.20  15498.80  15491.15  15496.85  2021-06-02   
96 2021-06-02 11:00:00  15497.10  15504.20  15496.10  15504.05  2021-06-02   
97 2021-06-02 11:05:00  15504.15  15511.60  15503.80  15506.05  2021-06-02   
98 2021-06-02 11:10:00  15506.00  15510.80  15503.90  15509.15  2021-06-02   
99 2021-06-02 11:15:00  15508.85  15514.10  15507.95  15512.20  2021-06-02   

         EMA_20  
0   15572.00000  
1   15572.00000  
2   15572

Signal Generation on Spot

In [8]:
import pandas as pd
from datetime import time

# === Load & Prepare Data ===
spot_5min = pd.read_parquet("nifty_spot_data_5min.parquet")
spot_5min = spot_5min.sort_values("Datetime").reset_index(drop=True)

# === Indicators ===
spot_5min['EMA_5'] = spot_5min['Close'].ewm(span=5, adjust=False).mean()
spot_5min['EMA_20'] = spot_5min['Close'].ewm(span=20, adjust=False).mean()
spot_5min['EMA_Cross_Up'] = (spot_5min['EMA_5'] > spot_5min['EMA_20']) & (spot_5min['EMA_5'].shift(1) <= spot_5min['EMA_20'].shift(1))
spot_5min['Above_Both_EMAs'] = (spot_5min['Close'] > spot_5min['EMA_5']) & (spot_5min['Close'] > spot_5min['EMA_20'])
#spot_5min['Next_Bullish'] = spot_5min['Close'].shift(-1) > spot_5min['Open'].shift(-1)
#spot_5min['Candle_Size_Small'] = (spot_5min['High'] - spot_5min['Low']) < 40

# Drop unstable rows
spot_5min = spot_5min.iloc[20:].dropna().reset_index(drop=True)

# === Trade Logic ===
open_trades = []
trade_log = []

for i in range(len(spot_5min) - 1):  # -1 to ensure next candle exists
    row = spot_5min.iloc[i]
    next_row = spot_5min.iloc[i + 1]
    current_time = row['Datetime'].time()

    # === Exit Logic ===
    for trade in open_trades[:]:
        entry_price = trade['Entry_Price']
        current_price = row['Close']
        pnl_pct = (current_price - entry_price) / entry_price

        if pnl_pct <= -0.005:
            trade_log.append({
                'Entry_Time': trade['Entry_Time'],
                'Entry_Price': entry_price,
                'Exit_Time': row['Datetime'],
                'Exit_Price': current_price,
                'PnL': current_price - entry_price,
                'Reason': 'Stop Loss -0.5%'
            })
            open_trades.remove(trade)

        elif pnl_pct >= 0.01:
            trade_log.append({
                'Entry_Time': trade['Entry_Time'],
                'Entry_Price': entry_price,
                'Exit_Time': row['Datetime'],
                'Exit_Price': current_price,
                'PnL': current_price - entry_price,
                'Reason': 'Take Profit +1%'
            })
            open_trades.remove(trade)

        elif current_time >= time(15, 15):
            trade_log.append({
                'Entry_Time': trade['Entry_Time'],
                'Entry_Price': entry_price,
                'Exit_Time': row['Datetime'],
                'Exit_Price': current_price,
                'PnL': current_price - entry_price,
                'Reason': 'EOD Exit'
            })
            open_trades.remove(trade)

    # === Entry Logic ===
    if (row['EMA_Cross_Up'] and row['Above_Both_EMAs']
        #and row['Candle_Size_Small'] and row['Next_Bullish']
        and time(9, 15) <= current_time <= time(14, 45)):
        
        open_trades.append({
            'Entry_Time': next_row['Datetime'],     # Enter on next candle open
            'Entry_Price': next_row['Open']
        })

# === Convert to DataFrame ===
trades_df = pd.DataFrame(trade_log)

# === Save Trades ===
trades_df.to_csv("ema_crossover_trades.csv", index=False)
trades_df.to_parquet("ema_crossover_trades.parquet", index=False)

# # === Summary ===
# print("\n Trade Summary:")
# print(trades_df if not trades_df.empty else "No trades executed.")
# print(f"\nTotal Trades: {len(trades_df)} | Net PnL: {trades_df['PnL'].sum():.2f}" if not trades_df.empty else "")

# === Summary ===
print("\n✅ Trade Summary:")
if not trades_df.empty:
    pd.set_option('display.max_rows', None)  # Show all rows
    pd.set_option('display.max_columns', None)  # Show all columns
    pd.set_option('display.width', 1000)  # Wide display
    print(trades_df)
    print(f"\nTotal Trades: {len(trades_df)} | Net PnL: {trades_df['PnL'].sum():.2f}")
else:
    print("No trades executed.")




✅ Trade Summary:
              Entry_Time  Entry_Price           Exit_Time  Exit_Price     PnL           Reason
0    2021-06-01 12:40:00     15581.40 2021-06-01 15:15:00    15580.65   -0.75         EOD Exit
1    2021-06-02 12:45:00     15500.00 2021-06-02 15:15:00    15587.20   87.20         EOD Exit
2    2021-06-02 13:55:00     15498.30 2021-06-02 15:15:00    15587.20   88.90         EOD Exit
3    2021-06-03 13:55:00     15652.60 2021-06-03 15:15:00    15693.35   40.75         EOD Exit
4    2021-06-04 12:35:00     15678.40 2021-06-04 15:15:00    15667.70  -10.70         EOD Exit
5    2021-06-07 12:30:00     15727.65 2021-06-07 15:15:00    15748.25   20.60         EOD Exit
6    2021-06-07 13:15:00     15736.95 2021-06-07 15:15:00    15748.25   11.30         EOD Exit
7    2021-06-08 09:20:00     15759.25 2021-06-08 15:15:00    15734.00  -25.25         EOD Exit
8    2021-06-08 11:10:00     15723.30 2021-06-08 15:15:00    15734.00   10.70         EOD Exit
9    2021-06-08 12:45:00     157

In [13]:
import pandas as pd
from datetime import time

# === Load and Prepare Data ===
df = pd.read_parquet("nifty_spot_data_5min.parquet")
df = df.sort_values("Datetime").reset_index(drop=True)
df['EMA_5'] = df['Close'].ewm(span=5, adjust=False).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
df['EMA_Cross_Up'] = (df['EMA_5'] > df['EMA_20']) & (df['EMA_5'].shift(1) <= df['EMA_20'].shift(1))
df['EMA_Cross_Down'] = (df['EMA_5'] < df['EMA_20']) & (df['EMA_5'].shift(1) >= df['EMA_20'].shift(1))
df['Above_Both_EMAs'] = (df['Close'] > df['EMA_5']) & (df['Close'] > df['EMA_20'])
df['Next_Bullish'] = df['Close'].shift(-1) > df['Open'].shift(-1)
df = df.iloc[20:].dropna().reset_index(drop=True)

# === Strategy Runner ===
def run_strategy(df, strategy_name, use_next_bullish=False, exit_on_cross_down=False):
    open_trades = []
    trade_log = []

    for i in range(len(df) - 1):
        row = df.iloc[i]
        next_row = df.iloc[i + 1]
        current_time = row['Datetime'].time()

        # Exit logic
        for trade in open_trades[:]:
            entry_price = trade['Entry_Price']
            current_price = row['Close']
            pnl_pct = (current_price - entry_price) / entry_price

            if pnl_pct <= -0.005:
                reason = "Stop Loss -0.5%"
            elif pnl_pct >= 0.01:
                reason = "Take Profit +1%"
            elif current_time >= time(15, 15):
                reason = "EOD Exit"
            elif exit_on_cross_down and row['EMA_Cross_Down']:
                reason = "Exit on EMA Cross Down"
            else:
                continue

            trade_log.append({
                'Datetime': row['Datetime'],
                'Entry_Time': trade['Entry_Time'],
                'Entry_Price': entry_price,
                'Exit_Time': row['Datetime'],
                'Exit_Price': current_price,
                'PnL': current_price - entry_price,
                'PnL_%': pnl_pct * 100,
                'Reason': reason,
                'Strategy': strategy_name
            })
            open_trades.remove(trade)

        # Entry logic
        if row['EMA_Cross_Up'] and row['Above_Both_EMAs'] and (time(9, 15) <= current_time <= time(14, 45)):
            if use_next_bullish and not row['Next_Bullish']:
                continue
            open_trades.append({
                'Entry_Time': next_row['Datetime'],
                'Entry_Price': next_row['Open']
            })

    trades_df = pd.DataFrame(trade_log)

    # Merge trade details back into full data
    full_df = df.copy()
    full_df['Trade_Entry'] = full_df['Datetime'].isin(trades_df['Entry_Time'].values)
    merged = pd.merge(
        full_df,
        trades_df[['Entry_Time', 'Exit_Time', 'Entry_Price', 'Exit_Price', 'PnL', 'Reason']],
        how='left',
        left_on='Datetime',
        right_on='Entry_Time'
    )
    merged.drop(columns=['Entry_Time'], inplace=True)

    return trades_df, merged

# === Strategy Variations ===
strategies = [
    ("Base", False, False),
    ("Strategy A (Exit on CrossDown)", False, True),
    ("Strategy B (Next Candle Bullish)", True, False),
    ("Strategy C (Bullish Entry + Exit on CrossDown)", True, True)
]

summary_rows = []
strategy_dfs = {}

for name, bullish, exit_down in strategies:
    trades, full_output = run_strategy(df, name, use_next_bullish=bullish, exit_on_cross_down=exit_down)
    strategy_dfs[name] = full_output
    if not trades.empty:
        summary_rows.append({
            'Strategy': name,
            'Total Trades': len(trades),
            'Total PnL': trades['PnL'].sum(),
            'Average PnL': trades['PnL'].mean(),
            'Win Rate (%)': (trades['PnL'] > 0).mean() * 100
        })

# === Save to Excel ===
summary_df = pd.DataFrame(summary_rows)

with pd.ExcelWriter("strategy_variation_results.xlsx") as writer:
    for name, df_ in strategy_dfs.items():
        df_.to_excel(writer, sheet_name=name[:31], index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("✅ Excel saved with full candle data + trade flags in 'strategy_variation_results.xlsx'")


✅ Excel saved with full candle data + trade flags in 'strategy_variation_results.xlsx'


In [4]:
print(trades_df.dtypes)

Entry_Time     datetime64[ns]
Entry_Price           float64
Exit_Time      datetime64[ns]
Exit_Price            float64
PnL                   float64
Reason                 object
dtype: object


OPTION PART

In [26]:
import pandas as pd

# Load Spot Trade Signals
df_trades = pd.read_parquet("ema_crossover_trades.parquet")
print("Spot Trade Signals:")
print(df_trades.head())

# Load Options Data
df_options = pd.read_parquet("nifty_options_data.parquet")
print("\nOptions Data:")
print(df_options.head())


Spot Trade Signals:
           Entry_Time  Entry_Price           Exit_Time  Exit_Price    PnL    Reason
0 2021-06-02 13:55:00     15498.30 2021-06-02 15:15:00    15587.20  88.90  EOD Exit
1 2021-06-03 13:55:00     15652.60 2021-06-03 15:15:00    15693.35  40.75  EOD Exit
2 2021-06-04 12:35:00     15678.40 2021-06-04 15:15:00    15667.70 -10.70  EOD Exit
3 2021-06-07 13:15:00     15736.95 2021-06-07 15:15:00    15748.25  11.30  EOD Exit
4 2021-06-08 12:45:00     15725.05 2021-06-08 15:15:00    15734.00   8.95  EOD Exit

Options Data:
         Date   Time  ExpiryDate StrikePrice Type     Open     High      Low    Close                           Ticker
0  2024-01-01  09:16  2024-01-04       20000   CE  1770.45  1770.45  1770.45  1770.45  2024010109:16NIFTY24010420000CE
1  2024-01-01  09:19  2024-01-04       20000   CE  1718.00  1718.00  1718.00  1718.00  2024010109:19NIFTY24010420000CE
2  2024-01-01  09:20  2024-01-04       20000   CE  1718.00  1718.00  1718.00  1718.00  2024010109:20NIFT

In [2]:
df_trades.columns = df_trades.columns.str.strip()  # Remove extra spaces
df_trades["Entry Price"] = df_trades["Entry_Price"].astype(float)  # Convert to numeric


In [3]:
# Initialize the options trade list
options_trades = []
print("Initialized options_trades list.")


Initialized options_trades list.


In [4]:
for index, row in df_trades.iterrows():
    trade_entry = {
        "Entry Time": row["Entry_Time"],
        "Exit Time": row["Exit_Time"]
    }
    options_trades.append(trade_entry)

print("Stored Entry Time and Exit Time for each trade.")


Stored Entry Time and Exit Time for each trade.


In [5]:
for i, row in enumerate(df_trades.itertuples(index=False)):
    option_type = "CE" if row.Type == "short" else "PE"
    options_trades[i]["Type"] = option_type

print("Stored Option Type (CE/PE) for each trade.")


Stored Option Type (CE/PE) for each trade.


In [6]:
if isinstance(options_trades, list):
    options_trades = pd.DataFrame(options_trades)

    options_trades["Reason"] = df_trades["Reason"].values
    options_trades["Trade Type"] = df_trades["Trade_Type"].values
    

In [7]:
print(options_trades.head(10))

           Entry Time           Exit Time Type           Reason Trade Type
0 2021-06-11 11:15:00 2021-06-11 14:00:00   PE  Exit Signal Hit   original
1 2021-06-11 12:00:00 2021-06-11 14:00:00   PE  Exit Signal Hit   original
2 2021-06-15 13:30:00 2021-06-15 15:15:00   PE         EOD Exit   original
3 2021-06-16 09:15:00 2021-06-16 13:45:00   PE  Exit Signal Hit    reentry
4 2021-06-16 10:15:00 2021-06-16 13:45:00   PE  Exit Signal Hit   original
5 2021-06-16 10:45:00 2021-06-16 13:45:00   PE  Exit Signal Hit   original
6 2021-06-16 12:30:00 2021-06-16 13:45:00   PE  Exit Signal Hit   original
7 2021-06-22 11:15:00 2021-06-22 15:15:00   PE         EOD Exit   original
8 2021-06-22 13:45:00 2021-06-22 15:15:00   PE         EOD Exit   original
9 2021-06-23 09:15:00 2021-06-23 13:45:00   PE  Exit Signal Hit    reentry


In [8]:
import pandas as pd

# Function to find the nearest multiple of n
def nearest_multiple(x, n):
    remainder = x % n
    if remainder < n / 2:
        nearest = x - remainder
    else:
        nearest = x - remainder + n
    return int(nearest)

# Ensure options_trades is a DataFrame
if isinstance(options_trades, list):
    options_trades = pd.DataFrame(options_trades)

# Apply ATM strike calculation
options_trades["StrikePrice"] = df_trades["Entry Price"].apply(lambda x: nearest_multiple(float(x), 50))

print("ATM Strike Prices calculated and stored.")
print(options_trades.tail())  # Display first few rows to verify


ATM Strike Prices calculated and stored.
              Entry Time           Exit Time Type           Reason Trade Type  \
1094 2025-03-27 09:15:00 2025-03-27 09:45:00   PE  Exit Signal Hit    reentry   
1095 2025-03-28 09:15:00 2025-03-28 14:15:00   PE  Exit Signal Hit   original   
1096 2025-03-28 09:30:00 2025-03-28 14:15:00   PE  Exit Signal Hit   original   
1097 2025-03-28 12:15:00 2025-03-28 14:15:00   PE  Exit Signal Hit   original   
1098 2025-03-28 12:30:00 2025-03-28 14:15:00   PE  Exit Signal Hit   original   

      StrikePrice  
1094        23550  
1095        23550  
1096        23500  
1097        23550  
1098        23500  


for rentries

In [9]:
print(options_trades.columns)  # Check available columns in df_trades

Index(['Entry Time', 'Exit Time', 'Type', 'Reason', 'Trade Type',
       'StrikePrice'],
      dtype='object')


In [19]:
# Step 1: Filter original EOD exits and get their StrikePrices in order
eod_strikes = options_trades[
    (options_trades['Reason'] == 'EOD Exit') & (options_trades['Trade Type'] == 'original')
]['StrikePrice'].tolist()

# Step 2: Find the reentry trades
reentry_indices = options_trades[
    options_trades['Trade Type'] == 'reentry'
].index

# Step 3: Assign the stored strike prices to reentry trades, one by one
for i, idx in enumerate(reentry_indices):
    if i < len(eod_strikes):
        options_trades.at[idx, 'StrikePrice'] = eod_strikes[i]
    else:
        # Optional: Handle case where there are more reentries than EOD exits
        options_trades.at[idx, 'StrikePrice'] = None  # or leave as-is

# ✅ Done


In [20]:
print(options_trades.head(11))

            Entry Time           Exit Time Type           Reason Trade Type  \
0  2021-06-11 11:15:00 2021-06-11 14:00:00   PE  Exit Signal Hit   original   
1  2021-06-11 12:00:00 2021-06-11 14:00:00   PE  Exit Signal Hit   original   
2  2021-06-15 13:30:00 2021-06-15 15:15:00   PE         EOD Exit   original   
3  2021-06-16 09:15:00 2021-06-16 13:45:00   PE  Exit Signal Hit    reentry   
4  2021-06-16 10:15:00 2021-06-16 13:45:00   PE  Exit Signal Hit   original   
5  2021-06-16 10:45:00 2021-06-16 13:45:00   PE  Exit Signal Hit   original   
6  2021-06-16 12:30:00 2021-06-16 13:45:00   PE  Exit Signal Hit   original   
7  2021-06-22 11:15:00 2021-06-22 15:15:00   PE         EOD Exit   original   
8  2021-06-22 13:45:00 2021-06-22 15:15:00   PE         EOD Exit   original   
9  2021-06-23 09:15:00 2021-06-23 13:45:00   PE  Exit Signal Hit    reentry   
10 2021-06-23 09:15:00 2021-06-23 13:45:00   PE  Exit Signal Hit    reentry   

    StrikePrice  
0         15800  
1         15750

In [21]:
options_trades.to_csv("final_spot_trades_with_strikes.csv", index=True)

In [22]:
import pandas as pd

# Ensure Date column in options data is datetime format
df_options["Date"] = pd.to_datetime(df_options["Date"])  
df_options["StrikePrice"] = df_options["StrikePrice"].astype(int)

# Loop through all trades
for index, row in options_trades.iterrows():
    entry_time = row["Entry Time"]
    strike_price = row["StrikePrice"]
    option_type = row["Type"]
    entry_date = entry_time.date()  # Convert to date only
    
    print(f"\n🔹 Trade {index + 1}: Entry Time {entry_time}, Strike {strike_price}, Type {option_type}")

    # Step 1: Filter by Strike Price
    filtered_1 = df_options[df_options["StrikePrice"] == strike_price]
    print(f"\nStep 1 - Matching Strike Price ({strike_price}): {len(filtered_1)} rows")
    print(filtered_1.head())

    # Step 2: Filter by Option Type
    filtered_2 = filtered_1[filtered_1["Type"] == option_type]
    print(f"\nStep 2 - Matching Type ({option_type}): {len(filtered_2)} rows")
    print(filtered_2.head())

    # Step 3: Filter by Entry Date
    filtered_3 = filtered_2[pd.to_datetime(filtered_2["Date"]) == pd.to_datetime(entry_date)]
    print(f"\nStep 3 - Matching Date ({entry_date}): {len(filtered_3)} rows")
    print(filtered_3.head())

    print("\n" + "-" * 80)  # Separator for readability



🔹 Trade 1: Entry Time 2021-06-11 11:15:00, Strike 15800, Type PE

Step 1 - Matching Strike Price (15800): 162831 rows
            Date   Time  ExpiryDate  StrikePrice Type   Open   High    Low  \
12817 2021-06-01  09:15  2021-06-03        15800   CE  13.70  13.70   9.00   
12818 2021-06-01  09:16  2021-06-03        15800   CE  13.10  13.10  11.65   
12819 2021-06-01  09:17  2021-06-03        15800   CE  11.90  11.95  10.90   
12820 2021-06-01  09:18  2021-06-03        15800   CE  11.75  12.65  11.10   
12821 2021-06-01  09:19  2021-06-03        15800   CE  12.10  12.10  11.35   

       Close                           Ticker  
12817  13.10  2021060109:15NIFTY21060315800CE  
12818  11.80  2021060109:16NIFTY21060315800CE  
12819  11.65  2021060109:17NIFTY21060315800CE  
12820  12.35  2021060109:18NIFTY21060315800CE  
12821  11.60  2021060109:19NIFTY21060315800CE  

Step 2 - Matching Type (PE): 117071 rows
            Date   Time  ExpiryDate  StrikePrice Type    Open    High     Low  \
1

In [23]:
# Ensure ExpiryDate column is in datetime format
df_options["ExpiryDate"] = pd.to_datetime(df_options["ExpiryDate"])
options_trades["NearestExpiry"] = None  # Add column to store expiry dates

# Iterate over all trades
for index, row in options_trades.iterrows():
    entry_time = pd.to_datetime(row["Entry Time"])  # Ensure datetime format
    strike_price = row["StrikePrice"]
    option_type = row["Type"]
    entry_date = entry_time.date()  # Convert to just date for filtering

    print(f"\n🔹 Trade {index + 1}: Entry Time {entry_time}, Strike {strike_price}, Type {option_type}")

    # Step 1: Filter by Strike Price
    filtered_1 = df_options[df_options["StrikePrice"] == strike_price]
    
    # Step 2: Filter by Type
    filtered_2 = filtered_1[filtered_1["Type"] == option_type]

    # Step 3: Filter by Date
    filtered_3 = filtered_2[pd.to_datetime(filtered_2["Date"]) == pd.to_datetime(entry_date)]
    
    # Ensure filtered_3 is a fresh copy to avoid SettingWithCopyWarning
    filtered_3 = filtered_3.copy()

    # Find the nearest expiry date that is on or after Entry Time
    nearest_expiry = filtered_3.loc[filtered_3["ExpiryDate"] >= entry_time, "ExpiryDate"].min()

    # If NaT, use the latest expiry for that entry date
    if pd.isna(nearest_expiry):
        nearest_expiry = filtered_3["ExpiryDate"].max()  # Latest expiry for the same day

    # Store the nearest expiry in the options_trades DataFrame
    options_trades.at[index, "NearestExpiry"] = nearest_expiry

    print(f"✅ Nearest Expiry: {nearest_expiry}")
    print("-" * 80)

print("\n🎯 All trades processed! Here's the updated DataFrame:")
print(options_trades.head())



🔹 Trade 1: Entry Time 2021-06-11 11:15:00, Strike 15800, Type PE
✅ Nearest Expiry: 2021-06-17 00:00:00
--------------------------------------------------------------------------------

🔹 Trade 2: Entry Time 2021-06-11 12:00:00, Strike 15750, Type PE
✅ Nearest Expiry: 2021-06-17 00:00:00
--------------------------------------------------------------------------------

🔹 Trade 3: Entry Time 2021-06-15 13:30:00, Strike 15850, Type PE
✅ Nearest Expiry: 2021-06-17 00:00:00
--------------------------------------------------------------------------------

🔹 Trade 4: Entry Time 2021-06-16 09:15:00, Strike 15850, Type PE
✅ Nearest Expiry: 2021-06-17 00:00:00
--------------------------------------------------------------------------------

🔹 Trade 5: Entry Time 2021-06-16 10:15:00, Strike 15800, Type PE
✅ Nearest Expiry: 2021-06-17 00:00:00
--------------------------------------------------------------------------------

🔹 Trade 6: Entry Time 2021-06-16 10:45:00, Strike 15800, Type PE
✅ Nearest

added time

In [12]:
import pandas as pd

# Assume full options_trades and df_options DataFrames are loaded

# Step 1: Convert Entry/Exit times and add 15 minutes
options_trades["Entry Time"] = pd.to_datetime(options_trades["Entry Time"], errors='coerce') + pd.Timedelta(minutes=15)
options_trades["Exit Time"] = pd.to_datetime(options_trades["Exit Time"], errors='coerce') + pd.Timedelta(minutes=15)
options_trades["NearestExpiry"] = pd.to_datetime(options_trades["NearestExpiry"], errors='coerce')

# Step 2: Add empty columns for prices
options_trades["EntryPrice"] = None
options_trades["ExitPrice"] = None

# Step 3: Preprocess df_options DateTime columns (only once for full data)
df_options["ExpiryDate"] = pd.to_datetime(df_options["ExpiryDate"], errors='coerce')
df_options["DateTime"] = pd.to_datetime(df_options["Date"].astype(str) + " " + df_options["Time"].astype(str), errors='coerce')

# Step 4: Loop through test trades and fetch prices
for index, row in options_trades.iterrows():
    entry_time = row["Entry Time"]
    exit_time = row["Exit Time"]
    expiry_date = row["NearestExpiry"]
    strike_price = row["StrikePrice"]
    option_type = row["Type"]

    filtered_df = df_options[
        (df_options["StrikePrice"] == strike_price) &
        (df_options["Type"] == option_type) &
        (df_options["ExpiryDate"] == expiry_date)
    ]

    entry_row = filtered_df[filtered_df["DateTime"] == entry_time]
    if not entry_row.empty:
        options_trades.at[index, "EntryPrice"] = entry_row.iloc[0]["Open"]

    exit_row = filtered_df[filtered_df["DateTime"] == exit_time]
    if not exit_row.empty:
        options_trades.at[index, "ExitPrice"] = exit_row.iloc[0]["Open"]

# Output results for testing
print(options_trades)
options_trades.to_csv("PE_options_trade.csv", index=False)


             Entry Time           Exit Time Type  StrikePrice NearestExpiry  \
0   2021-06-11 11:30:00 2021-06-11 14:15:00   PE        15800    2021-06-17   
1   2021-06-11 12:15:00 2021-06-11 14:15:00   PE        15750    2021-06-17   
2   2021-06-15 13:45:00 2021-06-15 15:30:00   PE        15850    2021-06-17   
3   2021-06-16 10:30:00 2021-06-16 14:00:00   PE        15800    2021-06-17   
4   2021-06-16 11:00:00 2021-06-16 14:00:00   PE        15800    2021-06-17   
..                  ...                 ...  ...          ...           ...   
782 2025-03-26 15:00:00 2025-03-26 15:30:00   PE        23500           NaT   
783 2025-03-28 09:30:00 2025-03-28 14:30:00   PE        23550           NaT   
784 2025-03-28 09:45:00 2025-03-28 14:30:00   PE        23500           NaT   
785 2025-03-28 12:30:00 2025-03-28 14:30:00   PE        23550           NaT   
786 2025-03-28 12:45:00 2025-03-28 14:30:00   PE        23500           NaT   

    EntryPrice ExitPrice  
0         99.0      89.0

the 10 min eod change


In [24]:
import pandas as pd

# Step 0: Load your data
# options_trades = pd.read_csv("your_options_trades_file.csv")
# df_options = pd.read_csv("your_options_chain_data_file.csv")

# Step 1: Convert Entry/Exit times and adjust
options_trades["Entry Time"] = pd.to_datetime(options_trades["Entry Time"], errors='coerce') + pd.Timedelta(minutes=15)

# Convert Exit Time first
options_trades["Exit Time"] = pd.to_datetime(options_trades["Exit Time"], errors='coerce')

# Apply conditional timedelta to Exit Time
def adjust_exit_time(dt):
    if pd.isna(dt):
        return dt
    if dt.time() == pd.to_datetime("15:15").time():
        return dt + pd.Timedelta(minutes=10)
    else:
        return dt + pd.Timedelta(minutes=15)

options_trades["Exit Time"] = options_trades["Exit Time"].apply(adjust_exit_time)
options_trades["NearestExpiry"] = pd.to_datetime(options_trades["NearestExpiry"], errors='coerce')

# Step 2: Add empty columns for prices
options_trades["EntryPrice"] = None
options_trades["ExitPrice"] = None

# Step 3: Preprocess df_options DateTime columns (only once for full data)
df_options["ExpiryDate"] = pd.to_datetime(df_options["ExpiryDate"], errors='coerce')
df_options["DateTime"] = pd.to_datetime(df_options["Date"].astype(str) + " " + df_options["Time"].astype(str), errors='coerce')

# Step 4: Loop through trades and fetch prices
for index, row in options_trades.iterrows():
    entry_time = row["Entry Time"]
    exit_time = row["Exit Time"]
    expiry_date = row["NearestExpiry"]
    strike_price = row["StrikePrice"]
    option_type = row["Type"]

    filtered_df = df_options[
        (df_options["StrikePrice"] == strike_price) &
        (df_options["Type"] == option_type) &
        (df_options["ExpiryDate"] == expiry_date)
    ]

    entry_row = filtered_df[filtered_df["DateTime"] == entry_time]
    if not entry_row.empty:
        options_trades.at[index, "EntryPrice"] = entry_row.iloc[0]["Open"]

    exit_row = filtered_df[filtered_df["DateTime"] == exit_time]
    if not exit_row.empty:
        options_trades.at[index, "ExitPrice"] = exit_row.iloc[0]["Open"]

# Step 5: Save output
options_trades.to_csv("rentry_PE_options_trade.csv", index=False)
print("Done. Output saved to rentry_PE_options_trade.csv")


Done. Output saved to rentry_PE_options_trade.csv


using dask

In [ ]:
import dask.dataframe as dd
import pandas as pd

# Step 1: Convert pandas DataFrames to Dask
ddf_trades = dd.from_pandas(options_trades.copy(), npartitions=4)
ddf_options = dd.from_pandas(df_options.copy(), npartitions=8)

In [ ]:


# Step 2: Convert datetime columns and adjust times
ddf_trades["Entry Time"] = dd.to_datetime(ddf_trades["Entry Time"], errors='coerce') + pd.Timedelta(minutes=15)
ddf_trades["Exit Time"] = dd.to_datetime(ddf_trades["Exit Time"], errors='coerce') + pd.Timedelta(minutes=15)
ddf_trades["NearestExpiry"] = dd.to_datetime(ddf_trades["NearestExpiry"], errors='coerce')

ddf_trades["EntryPrice"] = None
ddf_trades["ExitPrice"] = None

# Step 3: Preprocess options data DateTime
ddf_options["ExpiryDate"] = dd.to_datetime(ddf_options["ExpiryDate"], errors='coerce')
ddf_options["DateTime"] = dd.to_datetime(
    ddf_options["Date"].astype(str) + " " + ddf_options["Time"].astype(str), errors='coerce'
)

# Step 4: Convert back to pandas (if logic is too row-specific for Dask)
# (Dask is not great for row-wise logic like this, so we do this efficiently with filtered pandas chunks)

trades_pd = ddf_trades.compute()
options_pd = ddf_options.compute()

# Step 5: Match entry/exit prices efficiently
for index, row in trades_pd.iterrows():
    entry_time = row["Entry Time"]
    exit_time = row["Exit Time"]
    expiry = row["NearestExpiry"]
    strike = row["StrikePrice"]
    opt_type = row["Type"]

    # Filter once
    subset = options_pd[
        (options_pd["StrikePrice"] == strike) &
        (options_pd["Type"] == opt_type) &
        (options_pd["ExpiryDate"] == expiry)
    ]

    entry_row = subset[subset["DateTime"] == entry_time]
    if not entry_row.empty:
        trades_pd.at[index, "EntryPrice"] = entry_row.iloc[0]["Open"]

    exit_row = subset[subset["DateTime"] == exit_time]
    if not exit_row.empty:
        trades_pd.at[index, "ExitPrice"] = exit_row.iloc[0]["Open"]

# Final DataFrame result
print(trades_pd)
# Optionally export
trades_pd.to_csv("options_trade.csv", index=False)


KeyboardInterrupt: 

In [ ]:
options_trades["EntryPrice"] = None
options_trades["ExitPrice"] = None

for index, row in options_trades.iterrows():
    entry_time = pd.to_datetime(row["Entry Time"], errors='coerce')
    exit_time = pd.to_datetime(row["Exit Time"], errors='coerce')
    expiry_date = pd.to_datetime(row["NearestExpiry"], errors='coerce')
    strike_price = row["StrikePrice"]
    option_type = row["Type"]

    # Step 1: Filter by Strike Price, Type, and Expiry Date
    filtered_df = df_options[
        (df_options["StrikePrice"] == strike_price) &
        (df_options["Type"] == option_type) &
        (pd.to_datetime(df_options["ExpiryDate"], errors='coerce') == expiry_date)
    ]

    # Step 2: Create a single datetime column for comparison
    filtered_df["DateTime"] = pd.to_datetime(filtered_df["Date"].astype(str) + " " + filtered_df["Time"].astype(str), errors='coerce')

    # Step 3: Get Entry Price
    entry_row = filtered_df[filtered_df["DateTime"] == entry_time]
    if not entry_row.empty:
        options_trades.at[index, "EntryPrice"] = entry_row.iloc[0]["Open"]

    # Step 4: Get Exit Price
    exit_row = filtered_df[filtered_df["DateTime"] == exit_time]
    if not exit_row.empty:
        options_trades.at[index, "ExitPrice"] = exit_row.iloc[0]["Open"]

# Display the updated dataframe
print(options_trades.tail(10))

options_trades.to_csv("latest_w_options_trades.csv", index=False)



C:\Users\ALGO_12\AppData\Local\Temp\ipykernel_20760\2644768536.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["DateTime"] = pd.to_datetime(filtered_df["Date"].astype(str) + " " + filtered_df["Time"].astype(str), errors='coerce')
C:\Users\ALGO_12\AppData\Local\Temp\ipykernel_20760\2644768536.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["DateTime"] = pd.to_datetime(filtered_df["Date"].astype(str) + " " + filtered_df["Time"].astype(str), errors='coerce')
C:\Users\ALGO_12\A

              Entry Time           Exit Time Type  StrikePrice NearestExpiry  \
2831 2025-03-26 13:00:00 2025-03-27 09:45:00   PE        23550           NaT   
2832 2025-03-26 13:15:00 2025-03-27 09:45:00   PE        23550           NaT   
2833 2025-03-26 13:30:00 2025-03-27 09:45:00   PE        23500           NaT   
2834 2025-03-26 14:45:00 2025-03-27 09:45:00   PE        23500           NaT   
2835 2025-03-28 09:15:00 2025-03-28 10:45:00   PE        23550           NaT   
2836 2025-03-28 09:30:00 2025-03-28 10:45:00   PE        23500           NaT   
2837 2025-03-28 12:15:00 2025-03-28 14:15:00   PE        23550           NaT   
2838 2025-03-28 12:30:00 2025-03-28 14:15:00   PE        23500           NaT   
2839 2025-03-28 13:00:00 2025-03-28 14:15:00   PE        23500           NaT   
2840 2025-03-28 13:45:00 2025-03-28 14:15:00   PE        23450           NaT   

     EntryPrice ExitPrice  
2831       None      None  
2832       None      None  
2833       None      None  
2834   